# Flux post-processing chain: IRGA75 open-path correction with opco (2004-2017, 2019)

**notebook version**: `1` (28 Oct 2026)

## ℹ️ About this notebook

Corrects the CO2 flux of the **IRGA75** open-path gas analyser (LI-7500) with
[opco](https://github.com/holukas/opco), and stores the result in the external data folder as parquet.
opco models the bias of the open-path flux per half-hour as two physical terms, the heating of the
sensor (bulk-surface term) and the WPL thermal term, each scaled by a coefficient learned with
gradient-boosted trees. The coefficient of the WPL term also uses `k_op`, a weekly detector computed
from the LI-7500 data alone, so the model can be applied to years without the closed-path analyser.

The model is fitted here on the 2016-2017 overlap with the **IRGA72** closed-path analyser (LI-7200),
as the difference `FC` (IRGA75) minus `FC` (IRGA72) on the training half-hours. These are selected
here, from the same data versions: half-hours where the `FC` of both analysers passes Levels 2 to
3.2 of the chain, without a u* filter, and the LI-7200 flow rate is normal. The model is then applied
to every year of the input file, 2016, 2017 and 2019 included. The corrected flux is
`FC_L1.1_OPCO = FC - BIAS_OPCO`.

**The output is the input file of `00` with new columns added; no input column is changed.** `FC`
stays the uncorrected flux. The new columns are `FC_L1.1_OPCO` and the opco columns, which end in
`_OPCO`. They hold values only where `FC` holds one. Half-hours where a term of the bias cannot be
computed have no corrected flux.

### Known limitations

- 2016 and 2017 are the training years, so their correction is in sample.
- 2019 is the run `2019_3`, with another LI-7500 head than the one the model was trained on. Its
  correction rests on the assumption that the bias of that head behaves the same way.
- The two parts of the bias, `BIAS_HEAT_OPCO` and `BIAS_WPL_H_OPCO`, and the coefficients `XI_OPCO`
  and `K_OPCO` are not separately identified by day. Only their sum, `BIAS_OPCO`, is used.

***

## 📇 Data sources

- `30_FLUX_POST-PROCESSING_CHAIN/IRGA75/00_FPPC_IRGA75_INPUT_noSHC_2004-2017_2019.parquet`: the
  IRGA75 Level-1 fluxes without self-heating correction, joined with the meteo. It supplies the
  flux, the EddyPro columns and the meteo drivers of the model.
- `30_FLUX_POST-PROCESSING_CHAIN/IRGA72/00_FPPC_IRGA72_INPUT_2016-2026.09.parquet`: the IRGA72
  Level-1 fluxes joined with the meteo. It supplies the reference `FC` of the training, the
  LI-7200 flow rate and the columns its quality screening needs.

Both are addressed by absolute path and indexed on `TIMESTAMP_MIDDLE`.

**The quality screening of the training half-hours uses the Level 2 to 3.2 settings of the IRGA72
NEE notebook `IRGA72/01`**, for both analysers. The two differ only in the signal test (LI-7200
signal strength, LI-7500 AGC) and in the dates removed by hand, which exist for the LI-7200 only.
If the settings of `IRGA72/01` change, they have to be changed here too.

## ⚙️ Settings

In [1]:
# --- Sources -----------------------------------------------------------------------------------
DATADIR = (r"F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data"
           r"\workflow")
INFILE = rf"{DATADIR}\30_FLUX_POST-PROCESSING_CHAIN\IRGA75\00_FPPC_IRGA75_INPUT_noSHC_2004-2017_2019.parquet"
REFFILE = rf"{DATADIR}\30_FLUX_POST-PROCESSING_CHAIN\IRGA72\00_FPPC_IRGA72_INPUT_2016-2026.09.parquet"

# --- Output ------------------------------------------------------------------------------------
OUTNAME = "01_FPPC_IRGA75_INPUT_OPCO_2004-2017_2019"
OUTPATH = rf"{DATADIR}\30_FLUX_POST-PROCESSING_CHAIN\IRGA75"
FC_COL = 'FC'                  # the uncorrected IRGA75 flux, kept as it is
FC_CORR_COL = 'FC_L1.1_OPCO'   # the corrected IRGA75 flux
SUFFIX = '_OPCO'               # added to every opco column

# --- Model inputs --------------------------------------------------------------------------------
# Arguments of opco.physmodel.drivers() and the column of the input file each one reads. The flux
# run supplies the flux, its flag and the air properties; the meteo supplies the homogenised air
# temperature, SW_IN and VPD.
DRIVER_COLS = {
    'fc': 'FC', 'qc': 'FC_SSITC_TEST', 'h': 'H', 'ustar': 'USTAR', 'u': 'WS',
    'rho_c': 'CO2_MOLAR_DENSITY', 'rho_v': 'H2O_MOLAR_DENSITY', 'p': 'PA_EP',
    'rho_a': 'AIR_DENSITY', 'cp': 'AIR_CP',
    'ta': 'TA_T1_47_1_HOMOGENIZED_gfXG', 'sw_in': 'SW_IN_T1_47_1_gfXG', 'vpd': 'VPD_T1_47_1_HOMOGENIZED',
}
P_FACTOR = 1000  # PA_EP is in kPa, opco takes Pa

# The weekly detector leaves out half-hours with an AGC above AGC_MAX. The limit holds for the
# LI-7500 head of 2004-2017; the head of 2019 has another clean value, so it is not screened.
AGC_COL = 'CUSTOM_AGC_IRGA75_MEAN'
AGC_MAX = 50
AGC_SCREEN_END = '2017-12-31 23:45:00'
# Detector weeks run from this period middle (a Monday), so they do not depend on the first record.
WEEK_ORIGIN = '2004-01-05 00:15:00'

# --- Training ----------------------------------------------------------------------------------
TRAIN_START = '2016-01-01 00:15:00'
TRAIN_END = '2017-12-31 23:45:00'
SETUP = "CH-LAE LI-7500 75H-0640, overlap 2016-2017"

# Training half-hours: FC of both analysers accepted after Level 3.2 (QCF below ACCEPT_QCF_BELOW),
# and the LI-7200 flow rate at least FLOW_MIN. Level 2 to 3.2 settings as in IRGA72/01.
SITE_LAT = 47.478333  # CH-LAE
SITE_LON = 8.364389
UTC_OFFSET = 1  # timestamps are UTC+01:00 all year
NIGHTTIME_THRESHOLD = 20  # W m-2 potential radiation below this = nighttime
ACCEPT_QCF_BELOW = 2  # keep QCF 0 and 1, by day and by night
LEVEL2_SETTINGS = {
    'raw_data_screening_vm97': {'apply': True,
                                'spikes': True, 'amplitude': False, 'dropout': True,
                                'abslim': False, 'skewkurt_hf': False, 'skewkurt_sf': False,
                                'discont_hf': False, 'discont_sf': False},
    'ssitc': {'apply': True, 'setflag_timeperiod': None},
    'gas_completeness': {'apply': False},
    'spectral_correction_factor': {'apply': True},
    'angle_of_attack': {'apply': False, 'application_dates': []},
    'steadiness_of_horizontal_wind': {'apply': False},
}
SIGNAL_STRENGTH = {
    'IRGA72': {'apply': True, 'signal_strength_col': 'CUSTOM_SIGNAL_STRENGTH_IRGA72_MEAN',
               'method': 'discard below', 'threshold': 60},
    'IRGA75': {'apply': True, 'signal_strength_col': AGC_COL, 'method': 'discard above',
               'threshold': AGC_MAX},
}
CO2_CONC = 'CO2_MOLAR_DENSITY'  # the storage term is screened next to gaps in it (Level 3.1)
# Periods removed by hand in IRGA72/01 that fall in the training period.
REMOVE_DATES = {
    'IRGA72': [['2016-01-11', '2016-01-22'],   # LI-7200 installation, water in the filter
               ['2016-03-01', '2016-05-25']],  # too low a flow rate, water in the filter
    'IRGA75': [],
}
ABSLIM = (-50, 50)  # umol m-2 s-1
RANDUNC_COL = 'FC_RANDUNC_HF'
NIGHT_Z = 4  # night values below -NIGHT_Z times the pooled night spread are removed
FLOW_COL = 'CUSTOM_FLOW_VOLRATE_MEAN'  # LI-7200 flow rate, l min-1
FLOW_MIN = 8

# --- Output columns of opco --------------------------------------------------------------------
# Drivers and predictions kept in the output, with their names before SUFFIX is added.
OPCO_COLS = {'heat': 'HEAT', 'wpl_h': 'WPL_H', 'k_op': 'K_OP', 'xi': 'XI', 'k': 'K',
             'bias_heat': 'BIAS_HEAT', 'bias_wpl_h': 'BIAS_WPL_H', 'bias': 'BIAS',
             'out_of_envelope': 'OUT_OF_ENVELOPE'}

# umol CO2 m-2 s-1 over one half-hour, in gC m-2
GC_PER_HALFHOUR = 12.011e-6 * 1800

In [2]:
from datetime import datetime
from pathlib import Path
import importlib.metadata
import textwrap

import pandas as pd

from diive.core.io.files import save_parquet, load_parquet
from diive.flux.postprocessingchain import (init_flux_data, make_level32_detector, nighttime_uncertainty,
                                            run_level2, run_level31, run_level32)
from opco.physmodel import PhysBiasModel, drivers

pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 1000)


def tabcap(text, width=100):
    """Print a table caption, above the table it belongs to."""
    print(textwrap.fill(f"Table - {text}", width) + "\n")



# Start time is kept so the last cell can report the total runtime of the notebook.
NOTEBOOK_START = datetime.now()
print(f"Last run: {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
for _p in ('diive', 'opco', 'scikit-learn'):
    print(f"{_p} version: v{importlib.metadata.version(_p)}")

Last run: 2026-10-28 23:51:20
diive version: v0.93.0
opco version: v0.1.0
scikit-learn version: v1.8.0


## 📂 Load

In [3]:
df = load_parquet(filepath=INFILE)
ref = load_parquet(filepath=REFFILE).loc[TRAIN_START:TRAIN_END]

for _name, _d in [('IRGA75', df), ('IRGA72', ref)]:
    _steps = _d.index.to_series().diff().dropna().unique()
    assert _d.index.name == 'TIMESTAMP_MIDDLE' and not _d.index.duplicated().any()
    assert len(_steps) == 1 and _steps[0] == pd.Timedelta('30min'), f"{_name} is not on a continuous 30MIN index"
    assert _d.index[0] <= pd.Timestamp(TRAIN_START) and _d.index[-1] >= pd.Timestamp(TRAIN_END), (
        f"{_name} does not cover the training period")
    _need = [FC_COL, CO2_CONC, RANDUNC_COL, DRIVER_COLS['ta'], DRIVER_COLS['sw_in'], DRIVER_COLS['vpd'],
             SIGNAL_STRENGTH[_name]['signal_strength_col']] + ([FLOW_COL] if _name == 'IRGA72' else [])
    _missing = [c for c in _need if c not in _d.columns]
    assert not _missing, f"columns not in the {_name} file: {_missing}"
_missing = [c for c in DRIVER_COLS.values() if c not in df.columns]
assert not _missing, f"columns for opco not in the input file: {_missing}"
_new = [FC_CORR_COL] + [f"{c}{SUFFIX}" for c in OPCO_COLS.values()]
assert not set(_new) & set(df.columns), "an output column is already in the input file"

print(f"IRGA75  {len(df):>8,} records x {df.shape[1]} columns  {df.index[0]} -> {df.index[-1]}")
print(f"IRGA72  {len(ref):>8,} records x {ref.shape[1]} columns  {ref.index[0]} -> {ref.index[-1]}")

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROC
ESSING_CHAIN\IRGA75\00_FPPC_IRGA75_INPUT_noSHC_2004-2017_2019.parquet (0.354 seconds).

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROC
ESSING_CHAIN\IRGA72\00_FPPC_IRGA72_INPUT_2016-2026.09.parquet (0.275 seconds).

IRGA75   280,512 records x 339 columns  2004-01-01 00:15:00 -> 2019-12-31 23:45:00
IRGA72    35,088 records x 348 columns  2016-01-01 00:15:00 -> 2017-12-31 23:45:00


## 🎯 Training half-hours

In [4]:
def level32_accepted(d, name):
    """Return True where FC of analyser name passes Levels 2 to 3.2 (QCF accepted), without a u* filter."""
    d = d.drop(columns=[c for c in ('SW_IN_POT', 'DAYTIME', 'NIGHTTIME') if c in d.columns])
    data = init_flux_data(df=d, fluxcol=FC_COL, site_lat=SITE_LAT, site_lon=SITE_LON,
                          utc_offset=UTC_OFFSET, nighttime_threshold=NIGHTTIME_THRESHOLD,
                          daytime_accept_qcf_below=ACCEPT_QCF_BELOW,
                          nighttime_accept_qcf_below=ACCEPT_QCF_BELOW)
    data = run_level2(data, signal_strength=SIGNAL_STRENGTH[name], **LEVEL2_SETTINGS)
    data = run_level31(data, gapfill_storage_term=True, set_storage_to_zero=False,
                       screen_storage_term=True, concentration_col=CO2_CONC)
    data, sod = make_level32_detector(data)
    if REMOVE_DATES[name]:
        sod.flag_manualremoval_test(remove_dates=REMOVE_DATES[name], showplot=False, verbose=False)
        sod.addflag()
    sod.flag_outliers_abslim_test(minval=ABSLIM[0], maxval=ABSLIM[1], showplot=False, verbose=False)
    sod.addflag()
    sod.flag_outliers_hampel_test(window_length=48 * 13, n_sigma=5.5, use_differencing=True,
                                  separate_day_night=True, showplot=False, verbose=False, repeat=True)
    sod.addflag()
    sod.flag_outliers_hampel_test(window_length=48 * 13, n_sigma=5.5, use_differencing=False,
                                  by_time_of_day=True, time_of_day_slots=1, min_records=10,
                                  separate_day_night=True, showplot=False, verbose=False, repeat=True)
    sod.addflag()
    sigma = nighttime_uncertainty(data, sod, randunc_col=RANDUNC_COL, ta=DRIVER_COLS['ta'],
                                  vpd=DRIVER_COLS['vpd'], swin=DRIVER_COLS['sw_in'])
    sod.flag_outliers_negative_night_test(uncertainty=sigma['POOLED'], z=NIGHT_Z, showplot=False,
                                          verbose=False)
    sod.addflag()
    data = run_level32(data, outlier_detector=sod)
    return data.levels.filteredseries_level32_qcf.notna().reindex(d.index, fill_value=False)


_idx = ref.index
ok75 = level32_accepted(df.loc[TRAIN_START:TRAIN_END], 'IRGA75').reindex(_idx, fill_value=False)
ok72 = level32_accepted(ref, 'IRGA72').reindex(_idx, fill_value=False)
flow_ok = ref[FLOW_COL] >= FLOW_MIN
keep = ok75 & ok72 & flow_ok

──────────────────────────────────── Initializing flux data: FC ────────────────────────────────────

> Detected base variable CO2 for FC. (CO2 was used to calculate FC.)

> Potential radiation calculated (SW_IN_POT)

> Day/night flags: DAYTIME, NIGHTTIME

───────────────────────────────── Level 2: Quality Flag Expansion ──────────────────────────────────

...

> ++Added flag column FLAG_L3.2_NEE_L3.1_QCF_OUTLIER_HAMPEL_TEST to flag data

> ++Added flag column FLAG_L3.2_NEE_L3.1_QCF_OUTLIER_HAMPEL_2_TEST to flag data

Random uncertainty of the 10,009 nighttime records
uncertainty   median at night   below -2 sigma   below -3 sigma
───────────────────────────────────────────────────────────────
FS                       0.81   1,001 (10.0 %)      680 (6.8 %)
PAS20                    3.63       77 (0.8 %)        7 (0.1 %)
MAX                      3.66       72 (0.7 %)        6 (0.1 %)
POOLED                   2.83      205 (2.0 %)       83 (0.8 %)
FS: EddyPro, sampling error of each half-hour. PAS20: spread of similar measurements (ONEFlux), a 
generous estimate. MAX: the larger of the two. POOLED: spread of all nighttime values within 15 days
on either side.

> ++Added flag column FLAG_L3.2_NEE_L3.1_QCF_OUTLIER_NEGNIGHT_TEST to flag data

────────────────────────────── Level 3.2: Stepwise Outlier Detection ───────────────────────────────

### 📋 Table: training half-hours per screen

In [5]:
_tab = pd.DataFrame({'IRGA75 L3.2 accepted': ok75, 'IRGA72 L3.2 accepted': ok72,
                     f'LI-7200 flow >= {FLOW_MIN}': flow_ok, 'all three': keep})
_tab = _tab.groupby(_tab.index.year).sum().T
_tab['total'] = _tab.sum(axis=1)
tabcap("per year of the training period, the half-hours that pass each screen, and those that pass "
       "all three. The training uses the last row, less the half-hours where a bias term is missing")
display(_tab)

Table - per year of the training period, the half-hours that pass each screen, and those that pass
all three. The training uses the last row, less the half-hours where a bias term is missing



TIMESTAMP_MIDDLE,2016,2017,total
IRGA75 L3.2 accepted,8850,9404,18254
IRGA72 L3.2 accepted,9023,12314,21337
LI-7200 flow >= 8,12060,17169,29229
all three,5644,8149,13793


## 🧮 Model drivers

In [6]:
# The detector screen: AGC up to AGC_MAX until AGC_SCREEN_END, no AGC screen after it.
_ok = (df[AGC_COL] <= AGC_MAX) | (df.index > pd.Timestamp(AGC_SCREEN_END))
_s = {k: df[c].astype(float) for k, c in DRIVER_COLS.items()}
_s['p'] = _s['p'] * P_FACTOR
d = drivers(**_s, ok=_ok, origin=WEEK_ORIGIN)

_fc = d['fc'].notna()
print(f"{int(_fc.sum()):,} half-hours with FC; of these, "
      f"{int((_fc & d[['heat', 'wpl_h']].notna().all(axis=1)).sum()):,} have both bias terms and "
      f"{int((_fc & d['k_op'].isna()).sum()):,} lie in a week without a detector value")

219,044 half-hours with FC; of these, 200,560 have both bias terms and 42,637 lie in a week without a detector value


## 🎓 Fit

In [7]:
# Target: IRGA75 minus IRGA72 FC, on the training half-hours that have both terms.
_tr = d.loc[TRAIN_START:TRAIN_END].copy()
_tr['dfc'] = _tr['fc'] - ref[FC_COL].reindex(_tr.index)
train = keep.reindex(_tr.index, fill_value=False) & _tr[['dfc', 'heat', 'wpl_h']].notna().all(axis=1)

model = PhysBiasModel(setup=SETUP).fit(_tr[train])
_left = (_tr.loc[train, 'dfc'] - model.predict(_tr[train])['bias']).mean()
print(f"{model.trained['n']:,} training half-hours, {model.trained['start']} -> {model.trained['end']}")
print(f"mean IRGA75 - IRGA72: {_tr.loc[train, 'dfc'].mean():.3f} umol m-2 s-1, "
      f"after correction {_left:.3f} (in sample)")

13,792 training half-hours, 2016-01-23 14:15:00 -> 2017-12-12 08:15:00
mean IRGA75 - IRGA72: -1.850 umol m-2 s-1, after correction -0.021 (in sample)


## 🔧 Apply

In [8]:
_pred = d[['heat', 'wpl_h', 'k_op']].join(model.predict(d))
_has_fc = df[FC_COL].notna()

out = df.copy()
for _c, _name in OPCO_COLS.items():
    if _c == 'out_of_envelope':
        out[f"{_name}{SUFFIX}"] = _pred[_c] & _has_fc
    else:
        out[f"{_name}{SUFFIX}"] = _pred[_c].where(_has_fc)
out[FC_CORR_COL] = out[FC_COL] - out[f"BIAS{SUFFIX}"]
out = out.reindex(sorted(out.columns), axis=1)

print(f"{int(out[FC_CORR_COL].notna().sum()):,} of {int(_has_fc.sum()):,} FC values corrected, "
      f"{int(out[f'OUT_OF_ENVELOPE{SUFFIX}'].sum()):,} of them outside the training range")

200,560 of 219,044 FC values corrected, 4,296 of them outside the training range


### 📋 Table: correction per year

In [9]:
_y = out.index.year
_tab = pd.DataFrame({
    'FC_values': out[FC_COL].notna().groupby(_y).sum(),
    'corrected': out[FC_CORR_COL].notna().groupby(_y).sum(),
    'out_of_envelope': out[f"OUT_OF_ENVELOPE{SUFFIX}"].groupby(_y).sum(),
    'k_op_missing_%': (out[FC_COL].notna() & out[f"K_OP{SUFFIX}"].isna()).groupby(_y).sum()
                      .div(out[FC_COL].notna().groupby(_y).sum()).mul(100).round(1),
    'bias_gC': out[f"BIAS{SUFFIX}"].groupby(_y).sum().mul(GC_PER_HALFHOUR).round(1),
    'heat_part_gC': out[f"BIAS_HEAT{SUFFIX}"].groupby(_y).sum().mul(GC_PER_HALFHOUR).round(1),
    'h_part_gC': out[f"BIAS_WPL_H{SUFFIX}"].groupby(_y).sum().mul(GC_PER_HALFHOUR).round(1),
})
_tab.index.name = 'year'
tabcap("per calendar year, the half-hours with FC, those with a corrected flux, those outside the "
       "training range, the share of FC values in weeks without a detector value, and the sum of the "
       "bias and its two parts over the corrected half-hours, in gC m-2. The sums cover measured "
       "half-hours only, not a gap-filled year; a negative bias means the uncorrected flux is too low")
display(_tab[_tab['FC_values'] > 0])

Table - per calendar year, the half-hours with FC, those with a corrected flux, those outside the
training range, the share of FC values in weeks without a detector value, and the sum of the bias
and its two parts over the corrected half-hours, in gC m-2. The sums cover measured half-hours only,
not a gap-filled year; a negative bias means the uncorrected flux is too low



,FC_values,corrected,out_of_envelope,k_op_missing_%,bias_gC,heat_part_gC,h_part_gC
year,,,,,,,
2004,12011,11499,72,37.7,-333.1,-327.1,-5.5
2005,16486,15242,234,24.6,-467.6,-484.4,13.5
2006,16088,14953,164,27.2,-455.2,-468.3,1.7
2007,16438,14910,231,8.5,-429.9,-419.3,-58.1
2008,16305,14785,155,10.5,-508.6,-506.6,-31.8
2009,16186,14755,195,14.0,-535.6,-516.4,-37.5
2010,15387,13698,208,7.7,-491.8,-512.1,-13.1
2011,15795,14831,222,31.6,-590.4,-571.8,-67.3
2012,14612,12935,685,26.4,-482.6,-496.2,-25.0


## 🔍 Checks

In [10]:
assert out.index.equals(df.index), "the index changed"
assert set(out.columns) == set(df.columns) | set(_new), "a column was lost or added"
for _c in df.columns:
    assert out[_c].equals(df[_c]), f"input column {_c} changed"

# The corrected flux exists only where FC and the bias exist, and equals FC - BIAS there.
_corr = out[FC_CORR_COL].notna()
assert (_corr == (out[FC_COL].notna() & out[f"BIAS{SUFFIX}"].notna())).all()
assert (out.loc[_corr, FC_CORR_COL] == out.loc[_corr, FC_COL] - out.loc[_corr, f"BIAS{SUFFIX}"]).all()
# No opco value where there is no FC.
for _c in _new:
    if _c.startswith('OUT_OF_ENVELOPE'):
        assert not out.loc[~_has_fc, _c].any(), f"{_c} set where FC is missing"
    else:
        assert out.loc[~_has_fc, _c].isna().all(), f"{_c} has values where FC is missing"
print(f"Index, input columns and corrected flux verified. New columns: {', '.join(sorted(_new))}")

Index, input columns and corrected flux verified. New columns: BIAS_HEAT_OPCO, BIAS_OPCO, BIAS_WPL_H_OPCO, FC_L1.1_OPCO, HEAT_OPCO, K_OPCO, K_OP_OPCO, OUT_OF_ENVELOPE_OPCO, WPL_H_OPCO, XI_OPCO


## 💾 Export

In [11]:
Path(OUTPATH).mkdir(parents=True, exist_ok=True)
filepath = save_parquet(filename=OUTNAME, data=out, outpath=OUTPATH)
print(f"Saved to: {filepath}")

> Saved file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROC
ESSING_CHAIN\IRGA75\01_FPPC_IRGA75_INPUT_OPCO_2004-2017_2019.parquet (3.730 seconds).

Saved to: F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROCESSING_CHAIN\IRGA75\01_FPPC_IRGA75_INPUT_OPCO_2004-2017_2019.parquet


## ✅ Read the written file back

In [12]:
_written = load_parquet(filepath=str(Path(OUTPATH) / f"{OUTNAME}.parquet"))

print(f"index:   {_written.index[0]} -> {_written.index[-1]}  ({len(_written):,} records)")
print(f"columns: {len(_written.columns):,}")

assert _written.index.equals(out.index), "index changed on write"
assert list(_written.columns) == list(out.columns), "columns changed on write"
for _c in [FC_COL, 'LE', 'H'] + _new:
    assert _written[_c].equals(out[_c]), f"{_c} changed on write"
print("\nWritten file verified.")

> Loaded .parquet file 
F:\Sync\luhk_work\dev-data\datasets-data\dataset_ch-lae_flux_product-data\workflow\30_FLUX_POST-PROC
ESSING_CHAIN\IRGA75\01_FPPC_IRGA75_INPUT_OPCO_2004-2017_2019.parquet (0.372 seconds).

index:   2004-01-01 00:15:00 -> 2019-12-31 23:45:00  (280,512 records)
columns: 349

Written file verified.


***

# End of notebook.

In [13]:
NOTEBOOK_END = datetime.now()
_h, _rem = divmod(int((NOTEBOOK_END - NOTEBOOK_START).total_seconds()), 3600)
_m, _s = divmod(_rem, 60)

print(f"Started:  {NOTEBOOK_START.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Finished: {NOTEBOOK_END.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Runtime:  {_h:02d}:{_m:02d}:{_s:02d} (hh:mm:ss)")

Started:  2026-10-28 23:51:20
Finished: 2026-10-28 23:51:51
Runtime:  00:00:31 (hh:mm:ss)
